# Entropy as a Risk Regime Indicator: Information-Theoretic Analysis of Crypto and Equity Markets

**Author:** Anvitha  
**Date:** April 2026  
**Data:** BTC-USD, ETH-USD, S&P 500 (^GSPC) — January 2020 to December 2025  
**Language:** Python 3 (analysis); original thesis computed in R

---

## Abstract

> ✏️ **[WRITE THIS LAST — after you have results]**
>
> Template (fill in your actual findings):
>
> *This paper applies Shannon entropy as a time-varying measure of market complexity across Bitcoin (BTC), Ethereum (ETH), and the S&P 500 index over the period 2020–2025. Using a rolling window approach, we estimate normalised entropy from discretised log-return distributions and identify three distinct regimes: structured (low entropy), transitional (normal), and disordered (high entropy). We find that [YOUR FINDING — e.g. crypto markets exhibit persistently higher entropy than equities, with spikes co-incident with the Luna collapse (May 2022) and FTX bankruptcy (November 2022)]. Transfer entropy analysis reveals [YOUR FINDING — e.g. directional information flow from BTC to SP500 during crisis periods], confirmed by permutation tests at the 5% significance level. We interpret entropy as a risk regime indicator rather than a predictive signal, and discuss implications for adaptive risk management.*

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Mathematical Framework](#2-mathematical-framework)
   - 2.1 Shannon Entropy from First Principles
   - 2.2 Normalised Entropy and Financial Interpretation
   - 2.3 Transfer Entropy: Directed Information Flow
   - 2.4 Connection to the Efficient Market Hypothesis
3. [Data and Methodology](#3-data-and-methodology)
   - 3.1 Data Sources and Alignment
   - 3.2 Log Returns
   - 3.3 Discretisation (Binning)
   - 3.4 Rolling Window Design
4. [Results](#4-results)
   - 4.1 Rolling Entropy — Cross-Market Comparison
   - 4.2 Window Sensitivity Analysis
   - 4.3 Entropy vs Volatility
   - 4.4 Entropy Shock Detection
   - 4.5 Shock Streak Analysis
   - 4.6 Lead-Lag Probability Matrix
   - 4.7 Transfer Entropy and Net Information Flow
   - 4.8 Regime Classification
5. [Risk Interpretation](#5-risk-interpretation)
6. [Limitations](#6-limitations)
7. [References](#7-references)

---

## Imports and Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import yfinance as yf

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

COLOURS = {'BTC': '#F7931A', 'ETH': '#627EEA', 'SP500': '#1f77b4'}
EVENTS  = {
    '2020-03-16': 'COVID crash',
    '2022-05-09': 'Luna collapse',
    '2022-11-08': 'FTX bankruptcy',
}

print('Environment ready.')

---
## 1. Introduction

> ✏️ **[WRITE THIS — 3 to 4 short paragraphs. Suggested structure below:]**
>
> **Paragraph 1 — The problem:**  
> Standard risk measures like volatility capture the *magnitude* of price movements but not the *structural complexity* of a market. A market can be highly volatile yet predictable (e.g. a sustained trend), or low-volatility yet informationally disordered. This distinction matters for risk managers, not just traders.
>
> **Paragraph 2 — The tool:**  
> Shannon entropy, originally developed for communication theory, provides a natural measure of disorder in any probability distribution. Applied to the distribution of discretised returns over a rolling window, it yields a time-varying signal of how *unpredictable* a market's return distribution is — independent of the direction or magnitude of returns.
>
> **Paragraph 3 — This paper:**  
> We apply this framework to Bitcoin, Ethereum, and the S&P 500 over 2020–2025, a period covering multiple distinct stress regimes. We examine not only entropy dynamics within each market, but directional *information spillover* between markets using transfer entropy. Our goal is to characterise when markets become informationally disordered, and whether disorder spreads across asset classes.
>
> **Paragraph 4 — Scope:**  
> We deliberately frame entropy as a *risk regime indicator*, not an alpha signal. We do not claim predictive power. The honest contribution is descriptive and diagnostic.

---

---
## 2. Mathematical Framework

### 2.1 Shannon Entropy from First Principles

> ✏️ **[THIS IS YOUR DIFFERENTIATOR — do not skip or summarise. Write the full derivation.  
> Structure it exactly like this:]**

We seek a function $H$ that measures the *uncertainty* in a discrete probability distribution $\mathbf{p} = (p_1, p_2, \ldots, p_k)$. Rather than defining $H$ arbitrarily, we derive it from three axioms due to Khinchin (1957):

**Axiom 1 (Continuity):** $H$ is continuous in all $p_i$.  
**Axiom 2 (Maximality):** $H$ is maximised when $p_i = 1/k$ for all $i$ (uniform distribution).  
**Axiom 3 (Additivity):** $H$ satisfies a chain rule — the entropy of a joint experiment equals the entropy of one part plus the conditional entropy of the other.

> ✏️ **[ADD HERE: Walk through why these three axioms uniquely force the formula  
> $H = -\sum_{i} p_i \log p_i$.  
> Show each step. Explain what happens when $p_i = 0$ (the $0 \log 0 = 0$ convention).  
> This is the derivation. Do not just state the formula — prove it is the only possibility.  
> Reference: Shannon (1948), *A Mathematical Theory of Communication*.]**

---

### 2.2 Normalised Entropy and Financial Interpretation

For a discretisation into $B$ equiprobable bins, the maximum entropy is:

$$H_{\max} = \log B$$

We define **normalised entropy**:

$$H_N(t) = \frac{H(t)}{\log B} \in [0, 1]$$

> ✏️ **[ADD HERE: Explain the financial meaning of the endpoints.  
> $H_N = 1$: all bins equally populated — maximum uncertainty, returns are as unpredictable as possible.  
> $H_N = 0$: all mass in one bin — complete certainty, all returns are identical.  
> Explain *why* a disordered market (crisis period) pushes entropy toward 1 or collapses it toward 0 depending on the shock type. This is your original thinking.]**

**Implementation note:** The denominator must be $\log B$ (with the *same* base as the numerator), not $\log 10$ or any other fixed constant. In this notebook we use natural logarithms throughout, giving units of *nats*.

---

### 2.3 Transfer Entropy: Directed Information Flow

Standard correlation measures the *strength* of a linear relationship but not its *direction* in time, nor whether one variable genuinely reduces uncertainty about the other beyond its own past.

Transfer entropy from process $X$ to process $Y$, due to Schreiber (2000), is defined as:

$$T_{X \to Y} = \sum p(y_{t+1},\, y_t^{(l)},\, x_t^{(k)}) \log \frac{p(y_{t+1} \mid y_t^{(l)},\, x_t^{(k)})}{p(y_{t+1} \mid y_t^{(l)})}$$

where $y_t^{(l)}$ and $x_t^{(k)}$ denote the lag-$l$ and lag-$k$ histories respectively.

> ✏️ **[ADD HERE — this is the key derivation for interviews:]**  
> **Show that $T_{X \to Y}$ is a Kullback-Leibler divergence:**  
> $T_{X \to Y} = D_{KL}\bigl(p(y_{t+1} \mid y_t, x_t) \,\|\, p(y_{t+1} \mid y_t)\bigr)$  
> **Then explain why this means:**  
> - $T_{X \to Y} = 0$ if and only if $X$ adds *zero* information about $Y$'s future beyond $Y$'s own past  
> - $T_{X \to Y} \neq T_{Y \to X}$ in general — this asymmetry is what distinguishes it from correlation  
> - $T_{X \to Y} > 0$ does *not* imply causation in the structural sense, but it does imply Granger-type predictability  
> **This is exactly the question a quant interviewer will ask.**

---

### 2.4 Connection to the Efficient Market Hypothesis

> ✏️ **[ADD HERE — 1 paragraph. Key idea:**  
> Under the weak form EMH, past returns contain no exploitable information — the return distribution is as unpredictable as possible, which corresponds to **maximum entropy**.  
> If $H_N \approx 1$ persistently, the market is behaving consistently with EMH.  
> If $H_N$ drops sharply, the distribution of returns has become more concentrated — structure has emerged.  
> This gives entropy a theoretical grounding as an *EMH-consistency measure*, not just an empirical pattern.]**

---

---
## 3. Data and Methodology

### 3.1 Data Sources and Alignment

In [ ]:
# TODO: Download BTC-USD, ETH-USD, ^GSPC from Yahoo Finance
# Date range: 2020-01-01 to 2025-12-31
# Use closing prices only
# Align to S&P 500 trading calendar (drop weekends/holidays for BTC/ETH)
# Drop any remaining NaN rows
# Print shape check: should be ~1506 rows x 3 columns
pass

### 3.2 Log Returns

We compute continuously compounded log returns:

$$r_t = \log P_t - \log P_{t-1}$$

Log returns are preferred over simple returns for their time-additivity and approximate normality over short intervals.

In [ ]:
# TODO: Compute log returns for all three assets
# Drop the first row (NaN from diff)
# Confirm no NaN values remain
pass

### 3.3 Discretisation (Binning)

Shannon entropy requires a discrete probability distribution. We map continuous returns into $B = 10$ equiprobable bins using quantile-based breakpoints.

> **Design choice — Rolling quantiles vs full-sample quantiles:**  
> In this notebook we use full-sample quantile breakpoints for consistency with the thesis. This introduces a mild look-ahead bias in the bin boundaries, which is acceptable for descriptive analysis. For any application in a trading or risk context, breakpoints should be computed on an expanding or rolling window. See Section 6 (Limitations).

In [ ]:
# TODO: Implement quantile binning
# For each asset: compute 10 equiprobable bins using full-sample quantile breakpoints
# Assign each return to a bin label (1–10)
# Sanity check: each bin should contain ~10% of observations
pass

### 3.4 Rolling Window Design

We apply a rolling window of width $w = 100$ trading days (approximately 5 months). At each step $t$, we compute the empirical distribution over the $w$ most recent binned returns and calculate normalised Shannon entropy.

> **Window choice:** 100 days balances responsiveness to regime changes against noise in the empirical distribution. We validate this choice with a sensitivity analysis across $w \in \{10, 20, 50, 100, 200\}$ in Section 4.2.

In [ ]:
def shannon_entropy_normalised(binned_window: np.ndarray) -> float:
    """
    Compute normalised Shannon entropy for a window of discretised observations.

    Parameters
    ----------
    binned_window : array of integer bin labels

    Returns
    -------
    float in [0, 1]
        0 = all mass in one bin (complete certainty)
        1 = uniform distribution (maximum uncertainty)
    """
    # TODO: implement this function
    # 1. Count observations per bin
    # 2. Convert to probabilities
    # 3. Drop zero-probability bins (convention: 0 * log(0) = 0)
    # 4. Compute H = -sum(p * log(p))  [natural log]
    # 5. Normalise by log(number of non-empty bins)
    # 6. Return H_normalised
    pass

---
## 4. Results

### 4.1 Rolling Entropy — Cross-Market Comparison

In [ ]:
# TODO: Compute rolling entropy (window=100) for BTC, ETH, SP500
# Store in a DataFrame: columns = ['BTC', 'ETH', 'SP500'], index = date
# Plot: three lines on one chart, annotate COVID / Luna / FTX with vertical dashed lines
# Use COLOURS and EVENTS dicts defined in the imports cell
pass

> ✏️ **[WRITE 2–3 sentences after you see the chart: what do you observe?  
> e.g. Which market has the highest mean entropy? When do they converge?  
> What happens around FTX? Does entropy spike or collapse?]**

### 4.2 Window Sensitivity Analysis

In [ ]:
# TODO: Compute rolling entropy for BTC across window sizes [10, 20, 50, 100, 200]
# Plot all five on one chart
# Also compute: SD across window sizes at each date (the 'spread' plot from your R code)
# Interpretation: high spread = window choice is sensitive during that period
pass

### 4.3 Entropy vs Volatility

In [ ]:
# TODO: Compute rolling volatility (window=100) for all three assets
# Plot 1: entropy vs volatility scatter per asset (faceted, with regression line)
# Plot 2: time series of both signals overlaid for BTC
# Key question: do they move together or diverge? When do they diverge?
pass

> ✏️ **[KEY NARRATIVE POINT: If entropy and volatility diverge, that is your most interesting finding.  
> A period of high volatility but LOW entropy means: the market is moving a lot but in a *structured* way (e.g. a crash trend).  
> A period of low volatility but HIGH entropy means: the market is quiet but *informationally disordered* — harder to predict direction.  
> Find these periods in your data and name them.]**

### 4.4 Entropy Shock Detection

In [ ]:
# TODO: Compute first difference of entropy (day-over-day change)
# Flag shocks: days where the change falls outside the [5th, 95th] percentile
# Create shock_df: boolean columns for each asset, indexed by date
# Plot: heatmap of shocks across all three markets over time
pass

### 4.5 Shock Streak Analysis

In [ ]:
# TODO: Use run-length encoding to identify consecutive shock days per asset
# Compute: mean streak length, max streak length
# Present as a summary table
# Interpretation: longer streaks = more persistent structural disruption
pass

### 4.6 Lead-Lag Probability Matrix

In [ ]:
# TODO: Implement lead_lag_prob(leader, follower) function
# Logic: P(follower shock on day t+1 | leader shock on day t) vs baseline P(follower shock)
# Compute all 6 directional pairs: BTC<->ETH, BTC<->SP500, ETH<->SP500
# Store: direction, baseline prob, conditional prob, difference (lift)
# Plot: horizontal bar chart of lift for all 6 directions
pass

### 4.7 Permutation Tests (Significance of Lead-Lag)

In [ ]:
# TODO: Implement permutation_test(leader, follower, observed_diff, n_perm=10000)
# Under null hypothesis: shuffle leader randomly, recompute lift
# p-value = proportion of permutations where |permuted_diff| >= |observed_diff|
# Run for all 6 directions with set.seed equivalent: np.random.seed(42)
# Add p_value and significant (p < 0.05) columns to lead-lag table
# Print final table
pass

### 4.8 Transfer Entropy and Net Information Flow

In [ ]:
# TODO: Compute transfer entropy for all 6 directional pairs
# Use pyinform or a manual KSG estimator
# Bootstrap with n_boot=300 (minimum) for significance
# Set np.random.seed(42) for reproducibility
# Compute net flow: TE(X->Y) - TE(Y->X) for each pair
# Plot 1: heatmap of TE values (from row -> to column)
# Plot 2: bar chart of net flow by market (which is the dominant sender?)
pass

### 4.9 Regime Classification

In [ ]:
# TODO: Define three regimes per asset using IQR thresholds:
#   high   = entropy > 75th percentile
#   low    = entropy < 25th percentile
#   normal = otherwise
# For each asset: plot entropy time series coloured by regime
# Annotate COVID / Luna / FTX events
# Summary table: fraction of days in each regime per asset
pass

---
## 5. Risk Interpretation

> ✏️ **[WRITE THIS SECTION — it is what separates a research paper from a coding project.  
> Key points to make:]**
>
> **Entropy as a risk regime indicator, not a trading signal:**  
> We do not claim that high entropy predicts a crash. We claim it characterises the *current information environment*. A risk manager reading this would interpret persistent high entropy as a signal to reduce position sizing, widen confidence intervals on VAR estimates, and demand more conservative margin requirements — not to place directional bets.
>
> **The divergence between entropy and volatility matters:**  
> During periods where volatility is low but entropy is elevated, traditional risk models underestimate structural uncertainty. This is arguably the most useful finding for a risk desk.
>
> **Cross-market spillover implications:**  
> If your transfer entropy results show BTC leading SP500 during crisis periods, this has a concrete implication: crypto markets may serve as an early-warning signal for equity risk managers, even in a diversified portfolio that holds no crypto.
>
> **What this does NOT tell you:**  
> Entropy does not tell you the direction of the next move, the magnitude, or the timing. It is a state descriptor, not a forecast.

---

---
## 6. Limitations

> ✏️ **[WRITE ONE HONEST PARAGRAPH PER POINT. Do not hide these — stating them is a signal of intellectual maturity.]**

**L1 — Binning leakage:**  
Quantile breakpoints are computed on the full sample, which means the bin boundaries incorporate information from future dates. This is acceptable for descriptive analysis but would need to be replaced with expanding-window breakpoints in any live application.

**L2 — Window choice is arbitrary:**  
The choice of $w = 100$ days is motivated by interpretability (approximately one quarter) and validated through sensitivity analysis, but it is not derived from any theoretical criterion. Different windows yield qualitatively similar but quantitatively different regime labels.

**L3 — Transfer entropy is not causation:**  
Transfer entropy measures Granger-type predictability — whether $X$ reduces uncertainty about $Y$'s future beyond $Y$'s own past. It does not identify structural causation in the econometric sense. Confounders (common market-wide shocks) could produce non-zero TE without any direct channel.

**L4 — Alignment to equity calendar:**  
By aligning to S&P 500 trading days, we discard weekend and holiday returns for BTC and ETH. This may cause entropy to be underestimated for crypto during periods of heavy weekend activity (e.g. the crypto-specific crash dynamics of May 2022).

> ✏️ **[ADD any additional limitations specific to your findings here.]**

---

---
## 7. References

Shannon, C. E. (1948). A mathematical theory of communication. *Bell System Technical Journal*, 27(3), 379–423.

Khinchin, A. I. (1957). *Mathematical Foundations of Information Theory*. Dover Publications.

Schreiber, T. (2000). Measuring information transfer. *Physical Review Letters*, 85(2), 461–464.

Bentes, S. R., & Menezes, R. (2012). Entropy: A new measure of stock market volatility? *Journal of Physics: Conference Series*, 394, 012033.

Fiedor, P. (2014). Networks in financial markets based on the mutual information rate. *Physical Review E*, 89(5), 052801.

> ✏️ **[Add your thesis supervisor / any papers you actually cited in your thesis.]**